# RG3 CORAL 후보의 가스/오탐 전체 점검

104번 고정CORAL_all_CN7 후보: 전체 가스19개와 오탐정상26개 시각/외부모델점수/학습임계값/해당외부모델이 사용하는3개조건부정상순위 기록. 88번 탐지우선 후보8/61와 탐지 중복 비교. 사후개발 진단이며 외부 라벨로 threshold/행렬을 바꾸는 작업이 아님. 기록된순위와 모델 판정은 물리적 가스 원인 증거가 아니다.


In [1]:
from pathlib import Path
import numpy as np,pandas as pd
root=Path.cwd()
if not (root/'data').exists():root=root.parent
t=root/'reports/tables'
z=np.load(t/'rg3_F1_boost_fold_cache.npz',allow_pickle=False)
times=z['timestamps'];y=z['labels'];groups=z['groups'];blocks=np.unique(groups)
lookup={tuple(z[k].astype(int)):k.split('_')[1] for k in z.files if k.startswith('ref_')}
p=pd.read_csv(t/'rg3_CORAL_support_predictions.csv')
g=p[p.model=='CORAL_all_CN7_inner_F1_cut'].set_index('TimeStamp').loc[times].reset_index()
assert len(g)==620 and np.array_equal(g.actual,y)
ranks=np.empty((620,3))
for block in blocks:
 ref=np.flatnonzero(groups!=block);q=np.flatnonzero(groups==block);n=lookup[tuple(ref)]
 ranks[q]=z['features_'+n+'_1'][q][:,[6,8,9]]
g[['barrel2_noise_normal_rank','cushion_noise_normal_rank','barrel5_noise_normal_rank']]=ranks
old=pd.read_csv(t/'rg3_conditional_rank_predictions.csv')
old=old[(old.model=='conditional_rank75')&(old.budget==.1)][['TimeStamp','prediction']].rename(columns={'prediction':'previous_recall_candidate_prediction'})
g=g.merge(old,on='TimeStamp',validate='one_to_one')
g['score_minus_training_threshold']=g.score-g.threshold
g['new_gas_vs_previous_candidate']=(g.actual==1)&(g.prediction==1)&(g.previous_recall_candidate_prediction==0)
g['lost_gas_vs_previous_candidate']=(g.actual==1)&(g.prediction==0)&(g.previous_recall_candidate_prediction==1)
g[g.actual==1].to_csv(t/'rg3_CORAL_all19_gas_casebook.csv',index=False,encoding='utf-8-sig')
g[(g.actual==0)&(g.prediction==1)].to_csv(t/'rg3_CORAL_false26_casebook.csv',index=False,encoding='utf-8-sig')
print(g[(g.actual==1)&(g.prediction==1)][['TimeStamp','previous_recall_candidate_prediction']].to_string(index=False))
print('New gas',int(g.new_gas_vs_previous_candidate.sum()),'lost gas',int(g.lost_gas_vs_previous_candidate.sum()))
print('Casebook is retrospective development diagnosis, not physical cause confirmation.')


          TimeStamp  previous_recall_candidate_prediction
2020-10-22 04:46:36                                     1
2020-10-22 05:10:17                                     1
2020-10-23 05:15:46                                     1
2020-10-23 07:15:11                                     1
2020-11-04 05:17:45                                     1
2020-11-04 05:18:48                                     1
New gas 0 lost gas 2
Casebook is retrospective development diagnosis, not physical cause confirmation.
